# Manuscript figures
Computer Modern LaTeX typography, boxed axes, markers, dashed grids, and vector PDF export. Run from the repository root after generating the rank-test CSV files. A LaTeX distribution with type1cm, cm-super and dvipng is required.

## Plotting setup

In [ ]:
from pathlib import Path
import csv
import math
import numpy as np
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import brentq

D = Path.cwd() / 'figures'
D.mkdir(exist_ok=True)
mpl.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'axes.labelsize': 14,
    'axes.titlesize': 13,
    'font.size': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 300,
    'axes.linewidth': 1.0,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': .06,
})
BLUE, RED, GREEN, PURPLE = '#0072B2', '#B2182B', '#009E73', '#765099'
GREY, DARK = '#E8E8E8', '#222222'
c = 1/np.sqrt(2)

def axes_style(ax, grid=True):
    ax.set_axisbelow(True)
    if grid:
        ax.grid(True, which='major', linestyle='--', linewidth=.6, alpha=.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

def above(fig, handles, labels, ncol, y=.99):
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(.5,y),
               ncol=ncol, fontsize=11, frameon=True, framealpha=.6,
               edgecolor='black', facecolor='white', columnspacing=1.2,
               handletextpad=.5)

def save(fig, name):
    fig.savefig(D / (name + '.pdf'), dpi=300, bbox_inches='tight')
    plt.close(fig)

## Exact activation boundaries

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.65))
x = np.linspace(0, 1, 700)
y = np.sqrt(1-x*x)
a = axs[0]
a.fill_between(x,0,y,color=GREY)
a.fill_between(x,y,1,color=GREEN,alpha=.20)
a.plot(x,y,color=GREEN,lw=1.8)
a.text(.28,.27,'Compatible\n(no GMS)',ha='center',color=DARK,fontsize=11)
a.text(.77,.93,'GMS',ha='center',color=GREEN,fontsize=11)
a.text(.53,.73,r'$\eta_0^2+\eta_1^2=1$',rotation=-35,ha='center',va='center',fontsize=10,bbox=dict(fc='white',ec='none',alpha=.8,pad=1))
a.set(xlabel=r'Setting sharpness $\eta_0$',ylabel=r'Setting sharpness $\eta_1$',title='(a) One untrusted qubit')
a = axs[1]
a.axhspan(0,1,fc=GREY)
a.fill_between(x,c,1,where=x>=c,color=RED,alpha=.20)
xx = np.linspace(c,1,400)
a.fill_between(xx,c/xx,1,color=GREEN,alpha=.40)
a.plot(xx,c/xx,color=GREEN,lw=1.8)
a.axvline(c,color=RED,ls='--',lw=1.1)
a.axhline(c,color=RED,ls='--',lw=1.1)
t = 2**(-.25)
a.plot(t,t,'o',color=DARK,ms=4)
a.annotate(r'$(t,t)$',(t,t),(.59,.96),arrowprops={'arrowstyle':'-','lw':.8},fontsize=10)
a.plot(.75,.75,'s',color=RED,ms=4)
a.annotate('Both incompatible;\nno GMS',(.75,.75),(.32,.85),ha='center',va='center',arrowprops={'arrowstyle':'-','lw':.8},fontsize=9)
a.text(.32,.28,'Universally\nnon-GMS',ha='center',color=DARK,fontsize=11)
a.text(.94,.97,'GMS\npossible',ha='center',va='top',fontsize=9,color=DARK)
a.set(xlabel=r'Device sharpness $\eta_A$',ylabel=r'Device sharpness $\eta_B$',title='(b) Two untrusted qubits')
for a in axs:
    a.set(xlim=(0,1),ylim=(0,1),aspect='equal',xticks=[0,.5,c,1],yticks=[0,.5,c,1])
    a.set_xticklabels(['0','0.5',r'$1/\sqrt{2}$','1'])
    a.set_yticklabels(['0','0.5',r'$1/\sqrt{2}$','1'])
    axes_style(a,grid=False)
fig.tight_layout(w_pad=1.0)
save(fig,'activation')

## Graph-state schematics

In [ ]:
fig, axs = plt.subplots(1,3,figsize=(7.2,2.65))
def graph(ax,pos,edges,untrusted,title,labels=None):
    for u,v in edges:
        ax.plot([pos[u][0],pos[v][0]],[pos[u][1],pos[v][1]],color='#777777',lw=1.5,zorder=1)
    for u,(x,y) in enumerate(pos):
        ax.scatter(x,y,s=260,c=RED if u in untrusted else BLUE,edgecolors='white',linewidths=1.2,zorder=3)
        if labels is not None:
            ax.text(x,y,labels[u],ha='center',va='center',color='white',fontsize=10,zorder=4)
    ax.set_title(title,fontsize=12,y=1.02,pad=6)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.margins(.21)
    ax.set_box_aspect(1)
graph(axs[0],[(0,0),(1,0),(2,.6),(2,-.6),(3,.6),(3,-.6)],[(0,1),(1,2),(1,3),(2,3),(2,4),(3,5),(4,5)],{0,1},'(a) Leaf graph',[r'$\ell$',r'$p$','','','',''])
N=7
pos=[(np.cos(2*np.pi*i/N+np.pi/2),np.sin(2*np.pi*i/N+np.pi/2)) for i in range(N)]
graph(axs[1],pos,[(i,(i+1)%N) for i in range(N)],{0,1},'(b) Cycle graph',[str(i) for i in range(N)])
rng=np.random.default_rng(15)
N=9
pos=[(np.cos(2*np.pi*i/N+np.pi/2),np.sin(2*np.pi*i/N+np.pi/2)) for i in range(N)]
edges=[(i,j) for i in range(N) for j in range(i+1,N) if rng.random()<.5]
graph(axs[2],pos,edges,{0,4},'(c) Dense graph',[r'$a$','','','',r'$b$','','','',''])
fig.tight_layout(w_pad=.6)
save(fig,'graphs')

## Noise tolerance

In [ ]:
u = np.linspace(c,1,400)
g = 1-c/u
p = np.array([brentq(lambda z: .5*k*((1-z)**3+4*(1-z)**4+3*(1-z)**5)-2*np.sqrt(2),0,.2) if k>c else 0 for k in u])
pen = 4-2*np.sqrt(2)
gpen = 1-(2*np.sqrt(2)+pen)/(4*u+pen)
lam_star = 2*np.sqrt(2)-2
gvis = 1-2/(4*u-lam_star)
pvis = np.array([brentq(lambda z: .5*k*((1-z)**3+4*(1-z)**4+3*(1-z)**5)-lam_star*(1-z)**3-2,0,.2) if k>c else 0 for k in u])
fig, axs = plt.subplots(1,2,figsize=(7.2,3.85))
for ax, y, vis, ylab, title, ymax in zip(axs,[g,p],[gvis,pvis],[r'White-noise fraction $1-v$',r'Local depolarization $p$'],['(a) Global noise','(b) Independent local noise'],[.42,.115]):
    ax.fill_between(u,0,y,color=GREEN,alpha=.13)
    ax.plot(u,y,color=GREEN,lw=1.8,ls='-',marker='D',markersize=4,markevery=60,label='Projected (8 terms)')
    ax.plot(u,vis,color=PURPLE,lw=1.8,ls='-',marker='^',markersize=4,markevery=60,label='Visibility (9 terms)')
    ax.axvline(c,color='.45',ls='--',lw=.9)
    ax.text(.89,ymax*.12,'GMS certified',color=GREEN,ha='center',fontsize=10)
    ax.set(xlim=(.67,1.025),ylim=(0,ymax),xlabel=r'Sharpness product $u=\eta_A\eta_B$',ylabel=ylab,title=title,xticks=[c,.8,.9,1])
    ax.set_xticklabels([r'$c$','0.8','0.9','1'])
    for curve, color, offset in [(y,GREEN,(-7,-20 if ymax>.2 else -26)),(vis,PURPLE,(-6,8))]:
        ax.plot(1,curve[-1],'o',ms=4,color=color)
        ax.annotate(rf'{100*curve[-1]:.2f}\%',(1,curve[-1]),xytext=offset,textcoords='offset points',ha='right',color=color,fontsize=10,bbox=dict(fc='white',ec='none',alpha=.88,pad=.6))
    axes_style(ax)
axs[0].plot(u,gpen,color=BLUE,lw=1.8,ls='-.',marker='s',markerfacecolor='white',markersize=4,markevery=60,label='Penalty (5 terms)')
handles, labels = axs[0].get_legend_handles_labels()
order=[2,0,1]
above(fig,[handles[i] for i in order],[labels[i] for i in order],3)
fig.tight_layout(rect=(0,0,1,.88),w_pad=1.1)
save(fig,'noise')

## Circuit noise and matched shot-budget comparison

In [ ]:
import nbformat
_noise_nb = nbformat.read(Path.cwd()/'noise_budget.ipynb',as_version=4)
_noise_source = next(cell.source for cell in _noise_nb.cells if cell.cell_type=='code')
_noise_ns = {'__name__':'noise_budget'}
exec(compile(_noise_source,'noise_budget.ipynb','exec'),_noise_ns)
scores, bounds, weights = (_noise_ns[k] for k in ('scores','bounds','weights'))
q = np.linspace(0,.085,400)
vals = np.array([scores(z,.01) for z in q])
margin = vals-np.array(bounds)
fig, axs = plt.subplots(1,2,figsize=(7.2,3.85))
for i,(label,col,style,marker) in enumerate(zip(['Generator','Penalty','Projected','Visibility'],[RED,BLUE,GREEN,PURPLE],['--','-.','-',':'],['o','s','D','^'])):
    kws=dict(label=label,color=col,ls=style,lw=1.8,marker=marker,markersize=4,markevery=55,markerfacecolor='white' if i<2 else col)
    axs[0].plot(100*q,margin[:,i],**kws)
    positive=margin[:,i]>0
    shots=8*np.log(100)*weights[i]**2/margin[positive,i]**2
    axs[1].semilogy(100*q[positive],shots,**kws)
axs[0].axhline(0,color='.4',lw=.9)
axs[0].set(xlabel=r'Two-qubit depolarization $q$ (\%)',ylabel=r'Margin $F-B_F$',title='(a) Expected violation',xlim=(0,8.5))
axs[1].set(xlabel=r'Two-qubit depolarization $q$ (\%)',ylabel='Sufficient total shots',title='(b) Matched confidence and power',ylim=(1e2,1e7),xlim=(0,8.5))
for ax in axs:
    axes_style(ax)
handles, labels=axs[0].get_legend_handles_labels()
above(fig,handles,labels,4)
fig.tight_layout(rect=(0,0,1,.88),w_pad=1.1)
save(fig,'gatebudget')

## Analytical failure-probability bound

In [ ]:
ns=np.arange(8,71)
def bound(n):
    m=n-2
    sm=sum(math.comb(m,s)*(2**(s+2)-1)*2.**(-(m-s)) for s in (1,2))
    lg=sum(math.comb(m,s)*2.**(2*s-(s-2)*(m-s))/6 for s in range(3,m//2+1))
    return min(1,math.comb(n,2)*(3*2.**(-m)+sm+lg))
fig,ax=plt.subplots(figsize=(4.15,3.05))
ax.semilogy(ns,[bound(n) for n in ns],color=BLUE,lw=1.8,marker='o',markersize=4,markevery=7)
ax.set(xlabel=r'Number of vertices $n$',ylabel='Failure probability bound',ylim=(1e-12,1.6),xlim=(8,70),xticks=[10,20,30,40,50,60,70])
axes_style(ax)
fig.tight_layout()
save(fig,'random')

## Finite-size certificate frequencies

In [ ]:
with open(Path.cwd()/'random_summary.csv',newline='') as f:
    rows=list(csv.DictReader(f))
with open(Path.cwd()/'all_pair_extension.csv',newline='') as f:
    ext=list(csv.DictReader(f))
def wilson(k,n):
    p=k/n; z=1.95996398454; d=1+z*z/n
    a=(p+z*z/(2*n))/d
    w=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return p,max(0,a-w),min(1,a+w)
fig,ax=plt.subplots(figsize=(4.15,3.25))
for key,den,col,marker,label,style in [('fixed_pair_pass','samples',BLUE,'o','Designated pair','-'),('all_pair_pass','all_pair_samples',RED,'s','Every pair','--')]:
    data=[(int(r['n']),*wilson(int(r[key]),int(r[den]))) for r in rows if int(r[den])>0]
    if key=='all_pair_pass':
        for n in sorted({int(r['n']) for r in ext}):
            a=[int(r['all_pair_pass']) for r in ext if int(r['n'])==n]
            data.append((n,*wilson(sum(a),len(a))))
    x,y,lo,hi=np.array(data).T
    ax.errorbar(x,y,yerr=[np.maximum(0,y-lo),np.maximum(0,hi-y)],fmt=marker,linestyle=style,color=col,markerfacecolor=col if key=='fixed_pair_pass' else 'white',markersize=4,lw=1.8,capsize=2,label=label)
ax.set(xlabel=r'Number of vertices $n$',ylabel='Certificate frequency',ylim=(-.04,1.05),xticks=np.arange(6,25,3),xlim=(5.5,24.5))
axes_style(ax)
handles,labels=ax.get_legend_handles_labels()
above(fig,handles,labels,2)
fig.tight_layout(rect=(0,0,1,.88))
save(fig,'finite')